# Paper perturbation diagnostics: TEP

This notebook computes the perturbation-diagnostic tables used by the publication figure
`paper_side_by_side_state_jaccard_heatmaps_tep_fcc`. It intentionally contains no
episode-level inspection or exploratory plots.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Robust repository-root detection for execution from the repository root or notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from afc_robustness.diagnostics import (
    aggregate_diagnostic_metrics,
    dataset_perturbation_metrics,
)
from afc_robustness.experiment import BenchmarkConfig, load_dataset_from_config

## Load dataset and benchmark configuration

In [2]:
DATASET_LABEL = "TEP"
CONFIG_PATH = REPO_ROOT / "configs" / "tep.yaml"

cfg = BenchmarkConfig.from_yaml(CONFIG_PATH)
dataset = load_dataset_from_config(cfg)
bcfg = cfg.benchmark_config

print(f"Dataset: {dataset.name}")
print(f"Episodes: {dataset.n_episodes}")
print(f"Alarm tags: {dataset.n_tags}")
print(f"Time steps: {dataset.n_time_steps}")
print(f"Output directory: {cfg.output_dir}")

Dataset: tep
Episodes: 1000
Alarm tags: 50
Time steps: 60
Output directory: C:\Users\gianl\Documents\Code\afc-robustbench\results\tep


## Compute dataset-level perturbation diagnostics

In [3]:
SEVERITY_GRID = bcfg.get(
    "severity_grid",
    [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90],
)
N_DRAWS = int(bcfg.get("n_draws", 10))
INITIAL_ACTIVE_POLICY = str(bcfg.get("initial_active_policy", "act"))
ALLOW_LEADING_RTN = bool(bcfg.get("allow_leading_rtn", True))

# Use None for the full dataset. For quick checks, replace by a list of episode indices.
SAMPLE_INDICES = None

metrics_df = dataset_perturbation_metrics(
    dataset,
    cfg.perturbation_specs,
    severity_grid=SEVERITY_GRID,
    n_draws=N_DRAWS,
    sample_indices=SAMPLE_INDICES,
    random_seed=cfg.random_seed,
    dt=dataset.dt,
    initial_active_policy=INITIAL_ACTIVE_POLICY,
    allow_leading_rtn=ALLOW_LEADING_RTN,
    include_unrepaired=True,
    verbose=True,
    progress=True,
    progress_unit="sample",
)

agg_df = aggregate_diagnostic_metrics(metrics_df)

out_dir = cfg.output_dir / "diagnostics"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(out_dir / "perturbation_diagnostics_raw.csv", index=False)
agg_df.to_csv(out_dir / "perturbation_diagnostics_aggregated.csv", index=False)

print(f"Raw diagnostic rows: {len(metrics_df):,}")
print(f"Aggregated diagnostic rows: {len(agg_df):,}")
print(f"Saved diagnostics to: {out_dir}")

c:\Users\gianl\Documents\Code\afc-robustbench\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Perturbation diagnostics: dataset='tep', scenarios=6, severities=13, draws=10, samples=1000, variants=2, expected_rows=1,560,000.
Progress unit: sample; include_unrepaired=True; initial_active_policy='act'; allow_leading_rtn=True.


Perturbation diagnostics: 100%|██████████| 780000/780000 [37:02<00:00, 350.92sample/s, scenario=mixed_spurious_timing_late, severity=0.9, draw=10] 


Perturbation diagnostics finished: generated 1,560,000 rows.
Raw diagnostic rows: 1,560,000
Aggregated diagnostic rows: 156
Saved diagnostics to: C:\Users\gianl\Documents\Code\afc-robustbench\results\tep\diagnostics


## Sanity check for the paper diagnostic metric

In [4]:
PAPER_METRIC = "state_active_cell_jaccard_mean"
PAPER_VARIANT = "repaired"
PAPER_SCENARIOS = [
    "missing_events",
    "spurious_events",
    "timing_uncertainty",
    "late_time_delay",
    "mixed_missing_spurious",
    "mixed_spurious_timing_late",
]

check = agg_df.copy()
if "variant" in check.columns:
    check = check[check["variant"].astype(str).str.lower() == PAPER_VARIANT]
check = check[check["scenario"].astype(str).isin(PAPER_SCENARIOS)]

required_cols = ["scenario", "severity", PAPER_METRIC]
missing = [col for col in required_cols if col not in check.columns]
if missing:
    raise ValueError(f"Diagnostic table is missing columns required for the paper figure: {missing}")

check[required_cols].head(12)

,scenario,severity,state_active_cell_jaccard_mean
0,late_time_delay,0.00,1.000000
2,late_time_delay,0.05,0.331045
4,late_time_delay,0.10,0.253391
6,late_time_delay,0.15,0.206488
8,late_time_delay,0.20,0.156630
10,late_time_delay,0.25,0.122543
12,late_time_delay,0.30,0.090967
14,late_time_delay,0.40,0.037632
16,late_time_delay,0.50,0.012903
18,late_time_delay,0.60,0.007838
